# DiveSensei R41 PaliGemma Remote GPU Benchmark - Colab

Research-only visual proposal probe. This does not change `approve_review_v1`, taxonomy, auto-approval, or auto-exclusion.

Before running: set Runtime > Change runtime type > GPU. Add an HF token either in Colab Secrets as `HF_TOKEN`, or paste it into the environment manually.


In [ ]:
import os, subprocess, sys, tarfile
from pathlib import Path

print('Python', sys.version)
subprocess.run(['nvidia-smi'], check=False)


In [ ]:
!python -m pip install -q --upgrade pip
!python -m pip install -q torch transformers==4.53.3 accelerate sentencepiece huggingface-hub opencv-python-headless Pillow 'numpy<2'


In [ ]:
# Resolve Hugging Face token from Colab Secrets if available.
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        os.environ['HUGGINGFACE_HUB_TOKEN'] = token
        print('HF_TOKEN loaded from Colab userdata')
    else:
        print('HF_TOKEN is not set in Colab userdata')
except Exception as exc:
    print('Could not read Colab userdata:', repr(exc))

# Fallback if needed: uncomment and paste token manually, then delete notebook output.
# os.environ['HF_TOKEN'] = 'hf_...'
# os.environ['HUGGINGFACE_HUB_TOKEN'] = os.environ['HF_TOKEN']


In [ ]:
# Upload r41_remote_gpu_package.tar.gz to /content before this cell, or mount Drive and set PACKAGE_ARCHIVE.
PACKAGE_ARCHIVE = Path('/content/r41_remote_gpu_package.tar.gz')
if not PACKAGE_ARCHIVE.exists():
    from google.colab import files
    uploaded = files.upload()
    if 'r41_remote_gpu_package.tar.gz' not in uploaded:
        raise FileNotFoundError('Upload r41_remote_gpu_package.tar.gz')

with tarfile.open(PACKAGE_ARCHIVE, 'r:gz') as tar:
    tar.extractall('/content')
package_root = Path('/content/r41_remote_gpu_package')
print('Package root:', package_root)
print((package_root / 'REMOTE_PACKAGE_MANIFEST.json').read_text()[:1000])


In [ ]:
os.environ['HF_HOME'] = '/content/hf-cache'
cmd = [
    sys.executable, 'benchmarks/r41_remote_gpu_runner.py',
    '--cache-dir', os.environ['HF_HOME'],
    '--output-root', '/content/r41_remote_gpu_results',
    '--prompt-id', 'diving_attempt',
    '--decision-rule', 'yes_no_first_token_margin',
    '--smoke-max-frames', '1',
    '--full-fps', '1.0',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=package_root, check=True)


In [ ]:
summary = Path('/content/r41_remote_gpu_results/r41_remote_gpu_run_summary.md')
print(summary.read_text() if summary.exists() else 'missing summary')
from google.colab import files
files.download('/content/r41_remote_gpu_results_bundle.zip')
